In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from gwpy.timeseries import TimeSeries
from scipy import signal

sys.path.append("../src")
from senal import welch

In [ ]:
EVENTO = "GW150914"
DATOS = Path("../datos")
DURACION = 32
DURACION_FFT = 4
SEMILLA = 0

rng = np.random.default_rng(SEMILLA)

In [ ]:
fs = 4096
ruido = rng.normal(size=DURACION * fs)
frecuencias, psd = welch(ruido, fs, DURACION_FFT * fs)
_, psd_scipy = signal.welch(ruido, fs, nperseg=DURACION_FFT * fs)
print(np.abs(psd / psd_scipy - 1).max())
print(psd[1:-1].mean(), 2 / fs)

In [ ]:
plt.plot(frecuencias, psd, label="welch")
plt.axhline(2 / fs, color="k", label="2 / fs")
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("PSD (1/Hz)")
plt.legend();

In [ ]:
datos_h1 = TimeSeries.read(DATOS / f"H1_{EVENTO}_{DURACION}s.hdf5")
datos_l1 = TimeSeries.read(DATOS / f"L1_{EVENTO}_{DURACION}s.hdf5")
fs = int(datos_h1.sample_rate.value)
n_segmento = DURACION_FFT * fs

In [ ]:
frecuencias, media_h1 = welch(datos_h1.value, fs, n_segmento)
_, mediana_h1 = welch(datos_h1.value, fs, n_segmento, "mediana")
_, mediana_l1 = welch(datos_l1.value, fs, n_segmento, "mediana")

In [ ]:
_, scipy_media_h1 = signal.welch(datos_h1.value, fs, nperseg=n_segmento)
_, scipy_mediana_h1 = signal.welch(datos_h1.value, fs, nperseg=n_segmento, average="median")
asd_gwpy_h1 = datos_h1.asd(fftlength=DURACION_FFT)
print(np.abs(media_h1 / scipy_media_h1 - 1).max())
print(np.abs(mediana_h1 / scipy_mediana_h1 - 1).max())
print(np.abs(np.sqrt(mediana_h1) / asd_gwpy_h1.value - 1).max())

In [ ]:
plt.loglog(frecuencias, np.sqrt(mediana_h1), label="H1")
plt.loglog(frecuencias, np.sqrt(mediana_l1), label="L1")
plt.loglog(asd_gwpy_h1.frequencies.value, asd_gwpy_h1.value, "k--", linewidth=0.8, label="H1 (gwpy)")
plt.xlim(10, 2000)
plt.ylim(1e-24, 1e-19)
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("ASD (1/√Hz)")
plt.legend();

In [ ]:
for duracion in [1, 4, 16]:
    f, psd = welch(datos_h1.value, fs, duracion * fs, "mediana")
    plt.semilogy(f, np.sqrt(psd), label=f"{duracion} s")
plt.xlim(20, 80)
plt.ylim(1e-23, 1e-20)
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("ASD (1/√Hz)")
plt.legend();